<a href="https://colab.research.google.com/github/diarrandiaye0021-pixel/database_integration/blob/M/database_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Load the Datasets


In [ ]:
#import pandas as pd

import pandas as pd

In [ ]:
#Loads CityBikes data

bikes_url = "https://raw.githubusercontent.com/ljaycab/citybike-census/main/data/raw/citybike_networks_us.csv"

bikes_df = pd.read_csv(bikes_url)

bikes_df.head()

,network_id,name,city,state_abbr,n_stations,total_docks,pct_missing_capacity
0,citi-bike-nyc,Citi Bike,New York,NY,2467,73843.0,0.0
1,divvy,Divvy,Chicago,IL,2074,22423.0,0.0
2,capital-bikeshare,Capital BikeShare,Washington,DC,873,14795.0,0.0
3,bay-wheels,Bay Wheels,San Francisco Bay Area,CA,653,14294.0,0.0
4,blue-bikes,Blue Bikes,Boston,MA,625,10968.0,0.0


In [ ]:
#Load Census data

census_url = "https://raw.githubusercontent.com/ljaycab/citybike-census/main/data/raw/census_places.csv"

census_df = pd.read_csv(census_url, dtype={'geoid': str})

census_df.head()

,geoid,city,state,population,median_income,pct_bike_commute,pct_no_vehicle
0,3651000,New York,New York,8483844,80483.0,0.0159,0.5533
1,0644000,Los Angeles,California,3857263,81939.0,0.0059,0.1188
2,1714000,Chicago,Illinois,2711226,77902.0,0.0142,0.2659
3,4835000,Houston,Texas,2328253,64813.0,0.0048,0.1006
4,0455000,Phoenix,Arizona,1642323,81332.0,0.0040,0.0660


# Check the CityBikes Data

In [ ]:
#Count how many cities are missing a state

bikes_df['state_abbr'].isna().sum()

np.int64(0)

In [ ]:
#Counts how many records each city has

bikes_df['city'].value_counts()

,count
city,
New York,2
Portland,2
Washington,1
San Francisco Bay Area,1
Boston,1
Chicago,1
Philadelphia,1
Los Angeles,1
Milwaukee,1


In [ ]:
#Show the two NY rocords

bikes_df[bikes_df['city'] == 'New York']

,network_id,name,city,state_abbr,n_stations,total_docks,pct_missing_capacity
0,citi-bike-nyc,Citi Bike,New York,NY,2467,73843.0,0.0
32,joco-new-york,JOCO New York,New York,NY,35,915.0,0.0


In [ ]:
#Create a copy of the CityBikes data

bikes_clean = bikes_df.copy()

print("Original records:", len(bikes_df))
print("Working records:", len(bikes_clean))

Original records: 56
Working records: 56


# Combine Data from Both APIs

In [ ]:
# Remove Lime networks (no dock counts)
bikes_clean = bikes_clean[bikes_clean['pct_missing_capacity'] == 0]

# Combine networks in the same city, adding only stations and docks
bikes_clean = bikes_clean.groupby(['city', 'state_abbr'], as_index=False).agg(
    n_stations=('n_stations', 'sum'),
    total_docks=('total_docks', 'sum')
)

bikes_clean.head()

,city,state_abbr,n_stations,total_docks
0,Aspen,CO,101,1418.0
1,Austin,TX,87,1229.0
2,Aventura,FL,2,23.0
3,Beaverton,OR,26,1065.0
4,Bentonville,AR,30,420.0


In [ ]:
# State names
state_names = {
    'NY': 'New York', 'IL': 'Illinois', 'DC': 'District of Columbia',
    'CA': 'California', 'MA': 'Massachusetts', 'PA': 'Pennsylvania',
    'OR': 'Oregon', 'WI': 'Wisconsin', 'FL': 'Florida', 'HI': 'Hawaii',
    'CO': 'Colorado', 'TX': 'Texas', 'NE': 'Nebraska', 'MI': 'Michigan',
    'OH': 'Ohio', 'UT': 'Utah', 'IN': 'Indiana', 'TN': 'Tennessee',
    'AZ': 'Arizona', 'NV': 'Nevada', 'IA': 'Iowa', 'AR': 'Arkansas',
    'VA': 'Virginia', 'SC': 'South Carolina', 'ND': 'North Dakota',
}

In [ ]:
# Add a full state name column
bikes_clean['state'] = bikes_clean['state_abbr'].map(state_names)
bikes_clean.head()

,city,state_abbr,n_stations,total_docks,state
0,Aspen,CO,101,1418.0,Colorado
1,Austin,TX,87,1229.0,Texas
2,Aventura,FL,2,23.0,Florida
3,Beaverton,OR,26,1065.0,Oregon
4,Bentonville,AR,30,420.0,Arkansas


# Merge the Datasets

In [ ]:
merged = bikes_clean.merge(census_df, on=['city', 'state'], how='left')

# Rows with no geoid did not match
merged[merged['geoid'].isna()]

,city,state_abbr,n_stations,total_docks,state,geoid,population,median_income,pct_bike_commute,pct_no_vehicle
21,Honolulu,HI,133,2412.0,Hawaii,NaN,NaN,NaN,NaN,NaN
22,Indianapolis,IN,56,841.0,Indiana,NaN,NaN,NaN,NaN,NaN
24,Kailua-Kona,HI,23,360.0,Hawaii,NaN,NaN,NaN,NaN,NaN
34,Nashville,TN,36,419.0,Tennessee,NaN,NaN,NaN,NaN,NaN
45,San Francisco Bay Area,CA,653,14294.0,California,NaN,NaN,NaN,NaN,NaN


In [ ]:
census_df[census_df['city'].str.contains('Honolulu|Indianapolis|Nashville|San Francisco', case=False)]

,geoid,city,state,population,median_income,pct_bike_commute,pct_no_vehicle
15,1836003,Indianapolis city (balance),Indiana,885860,66219.0,0.0035,0.0841
16,0667000,San Francisco,California,830235,140970.0,0.0330,0.3016
20,4752006,Nashville-Davidson metropolitan government (ba...,Tennessee,690130,77371.0,0.0034,0.0665
54,1571550,Urban Honolulu,Hawaii,345482,86504.0,0.0184,0.1661
654,0673262,South San Francisco,California,64305,136578.0,0.0080,0.0752
855,1506290,East Honolulu,Hawaii,51360,159518.0,0.0044,0.0333
1139,1523150,Kailua CDP (Honolulu County),Hawaii,39640,148582.0,0.0097,0.0320
6276,3746000,Nashville,North Carolina,5729,61000.0,0.0000,0.1185
6887,1354264,Nashville,Georgia,4928,42516.0,0.0198,0.0751
7772,0548560,Nashville,Arkansas,4063,41142.0,0.0017,0.0999


In [ ]:
# Make the bike city names match the Census names
bikes_clean = bikes_clean.replace({'city': {
    'San Francisco Bay Area': 'San Francisco',
    'Honolulu': 'Urban Honolulu',
    'Indianapolis': 'Indianapolis city (balance)',
    'Nashville': 'Nashville-Davidson metropolitan government (balance)',
}})

# Drop Kailua-Kona (too small to compare)
bikes_clean = bikes_clean[bikes_clean['city'] != 'Kailua-Kona']

bikes_clean.shape

(51, 5)

In [ ]:
merged = bikes_clean.merge(census_df, on=['city', 'state'], how='left')

# Should show no rows if everything matched
merged[merged['geoid'].isna()]

,city,state_abbr,n_stations,total_docks,state,geoid,population,median_income,pct_bike_commute,pct_no_vehicle


In [ ]:
print("Cities:", len(merged))
print("Duplicate cities:", merged['geoid'].duplicated().sum())

# Count missing values in each column
merged.isna().sum()

Cities: 51
Duplicate cities: 0


,0
city,0
state_abbr,0
n_stations,0
total_docks,0
state,0
geoid,0
population,0
median_income,0
pct_bike_commute,0
pct_no_vehicle,0


In [ ]:
# Check data types
merged.dtypes

,0
city,object
state_abbr,object
n_stations,int64
total_docks,float64
state,object
geoid,object
population,int64
median_income,float64
pct_bike_commute,float64
pct_no_vehicle,float64


# Save to SQLite

In [ ]:
import sqlite3

# Cutoff: only cities with at least this many residents
MIN_POPULATION = 250000

major = merged[merged['population'] >= MIN_POPULATION]

# Open the database file (creates bike_share.db if it doesn't exist yet)
conn = sqlite3.connect("bike_share.db")

# Save the tables (all 51 cities stay in city_combined)
bikes_df.to_sql("citybike_networks", conn, if_exists="replace", index=False)
census_df.to_sql("census_places", conn, if_exists="replace", index=False)
merged.to_sql("city_combined", conn, if_exists="replace", index=False)
major.to_sql("city_major", conn, if_exists="replace", index=False)

# Check that everything saved
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
print(pd.read_sql("SELECT COUNT(*) AS all_cities FROM city_combined", conn))
print(pd.read_sql("SELECT COUNT(*) AS major_cities FROM city_major", conn))

conn.close()

                name
0  citybike_networks
1      census_places
2      city_combined
3         city_major
   all_cities
0          51
   major_cities
0            29
